# coord_verification for ps1

In [1]:

import numpy as np
import torch
from pathlib import Path
import json
import struct
from PIL import Image
import os

# ============================================================================
# Part 1: Calibration Data Generator
# ============================================================================

class CalibrationDataGenerator:
    """既知の3D座標を持つ合成シーンデータの生成"""
    
    def __init__(self, num_views=4, num_points=100):
        self.num_views = num_views
        self.num_points = num_points
        
    def generate_cube_points(self, size=1.0):
        """立方体形状の3D点群を生成"""
        points = []
        
        # 立方体の8頂点
        for x in [-size, size]:
            for y in [-size, size]:
                for z in [-size, size]:
                    points.append([x, y, z])
        
        # 辺上の点
        n_edge = max(1, (self.num_points - 8) // 12)
        for i in range(n_edge):
            t = (i + 1) / (n_edge + 1)
            points.extend([
                [2*size*t - size, -size, -size],
                [2*size*t - size, size, -size],
                [2*size*t - size, -size, size],
                [2*size*t - size, size, size],
                [-size, 2*size*t - size, -size],
                [size, 2*size*t - size, -size],
                [-size, 2*size*t - size, size],
                [size, 2*size*t - size, size],
                [-size, -size, 2*size*t - size],
                [size, -size, 2*size*t - size],
                [-size, size, 2*size*t - size],
                [size, size, 2*size*t - size],
            ])
        
        return np.array(points[:self.num_points])
    
    def generate_circular_camera_poses(self, radius=5.0, height=0.0):
        """円形配置のカメラポーズを生成"""
        poses = []
        for i in range(self.num_views):
            angle = 2 * np.pi * i / self.num_views
            
            cam_pos = np.array([
                radius * np.cos(angle),
                radius * np.sin(angle),
                height
            ])
            
            forward = -cam_pos / np.linalg.norm(cam_pos)
            up = np.array([0, 0, 1])
            right = np.cross(up, forward)
            right = right / np.linalg.norm(right)
            up = np.cross(forward, right)
            
            R = np.stack([right, up, forward], axis=0)
            t = -R @ cam_pos
            
            pose = np.eye(4)
            pose[:3, :3] = R
            pose[:3, 3] = t
            
            poses.append(pose)
            
        return poses
    
    def create_mock_scene(self, device='cpu'):
        """MASt3R風のシーンオブジェクトを作成"""
        
        pts3d_world = self.generate_cube_points(size=1.0)
        camera_poses = self.generate_circular_camera_poses(radius=5.0, height=0.5)
        
        class MockScene:
            def __init__(self, pts3d_world, camera_poses, device):
                self.device = device
                self.num_views = len(camera_poses)
                self.num_points = len(pts3d_world)
                
                self.ground_truth_pts3d_world = pts3d_world.copy()
                self.ground_truth_poses = [p.copy() for p in camera_poses]
                
                self.imgs = []
                for i in range(self.num_views):
                    self.imgs.append({
                        'idx': i,
                        'instance': f'view_{i:02d}',
                        'true_shape': np.array([512, 512])
                    })
                
                self.pts3d = []
                self.conf = []
                for pose in camera_poses:
                    pts_cam = self._world_to_camera(pts3d_world, pose)
                    self.pts3d.append(torch.from_numpy(pts_cam).float().to(device))
                    self.conf.append(torch.ones(self.num_points).float().to(device))
                
                self.focals = torch.tensor([[500.0]] * self.num_views).to(device)
                self.principal_points = torch.tensor([[256.0, 256.0]] * self.num_views).to(device)
                
                # Camera-to-world poses (C2W)
                self.im_poses = torch.stack([
                    torch.eye(4) for _ in range(self.num_views)
                ]).float().to(device)
                
                for i, pose in enumerate(camera_poses):
                    # W2CをC2Wに変換
                    c2w = torch.from_numpy(np.linalg.inv(pose)).float()
                    self.im_poses[i] = c2w
            
            def _world_to_camera(self, pts_world, pose):
                pts_homo = np.hstack([pts_world, np.ones((len(pts_world), 1))])
                pts_cam_homo = (pose @ pts_homo.T).T
                return pts_cam_homo[:, :3]
            
            def get_im_poses(self):
                return self.im_poses
            
            def get_pts3d(self):
                """Traditional Methodはlistを期待"""
                return self.pts3d
            
            def get_conf(self, i=None):
                return self.conf[i] if i is not None else self.conf
            
            def get_focals(self):
                return self.focals
            
            def get_principal_points(self):
                return self.principal_points
        
        scene = MockScene(pts3d_world, camera_poses, device)
        return scene
    
    def save_ground_truth(self, scene, output_path):
        """Ground truthデータを保存"""
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        
        np.savetxt(
            output_path / 'ground_truth_points3d.txt',
            scene.ground_truth_pts3d_world,
            header='X Y Z (world coordinates)',
            fmt='%.6f'
        )
        
        with open(output_path / 'ground_truth_poses.json', 'w') as f:
            poses_list = [p.tolist() for p in scene.ground_truth_poses]
            json.dump({
                'poses': poses_list,
                'description': '4x4 transformation matrices (world to camera)'
            }, f, indent=2)
        
        print(f"✓ Ground truth saved to {output_path}")

# process1

In [2]:

# ===== Traditional Method: extract_colmap_data =====
def extract_colmap_data_traditional(scene, image_paths, max_points=1000000):
    """
    Traditional Method: Extract COLMAP-compatible data from a MASt3R scene.
    (Derived from dino-mast3r-gs-kg-34oo.ipynb)
    """
    print("\n=== [TRADITIONAL] Extracting COLMAP-compatible data ===")

    # Extract point cloud
    pts_all = scene.get_pts3d()
    print(f"pts_all type: {type(pts_all)}")

    if isinstance(pts_all, list):
        print(f"pts_all is a list with {len(pts_all)} elements")
        if len(pts_all) > 0:
            print(f"First element type: {type(pts_all[0])}")
            if hasattr(pts_all[0], 'shape'):
                print(f"First element shape: {pts_all[0].shape}")

        pts_all = torch.stack([p if isinstance(p, torch.Tensor) else torch.tensor(p)
                              for p in pts_all])
        print(f"pts_all shape after conversion: {pts_all.shape}")

    if len(pts_all.shape) == 4:
        print(f"Found batched point cloud: {pts_all.shape}")
        B, H, W, _ = pts_all.shape
        pts3d = pts_all.reshape(-1, 3).detach().cpu().numpy()

        # Extract colors
        colors = []
        for img_path in image_paths:
            img = Image.open(img_path).resize((W, H))
            colors.append(np.array(img))
        colors = np.stack(colors).reshape(-1, 3) / 255.0
    else:
        pts3d = pts_all.detach().cpu().numpy() if isinstance(pts_all, torch.Tensor) else pts_all
        colors = np.ones((len(pts3d), 3)) * 0.5

    print(f"✓ Extracted {len(pts3d)} 3D points from {len(image_paths)} images")

    # Downsample points
    if len(pts3d) > max_points:
        print(f"\n⚠ Downsampling from {len(pts3d)} to {max_points} points...")
        valid_mask = ~(np.isnan(pts3d).any(axis=1) | np.isinf(pts3d).any(axis=1))
        pts3d_valid = pts3d[valid_mask]
        colors_valid = colors[valid_mask]
        
        # Count excluded points
        num_excluded = len(pts3d_valid) - max_points
        
        indices = np.random.choice(len(pts3d_valid), size=max_points, replace=False)
        pts3d = pts3d_valid[indices]
        colors = colors_valid[indices]
        print(f"✓ Downsampled to {len(pts3d)} points")
        print(f"⚠ Excluded {num_excluded} points due to max_points limit")

    # Extract camera parameters
    print("Extracting camera parameters...")

    # [Important] Convert camera-to-world (C2W) to world-to-camera (W2C)
    poses_c2w = scene.get_im_poses().detach().cpu().numpy()
    print(f"Retrieved camera-to-world poses: shape {poses_c2w.shape}")

    poses = []
    for i, pose_c2w in enumerate(poses_c2w):
        pose_w2c = np.linalg.inv(pose_c2w)
        poses.append(pose_w2c)
    poses = np.array(poses)
    print("Converted to world-to-camera poses for COLMAP")

    focals = scene.get_focals().detach().cpu().numpy()
    pp = scene.get_principal_points().detach().cpu().numpy()
    print(f"Focals shape: {focals.shape}")
    print(f"Principal points shape: {pp.shape}")

    mast3r_size = 224.0

    cameras = []
    for i, img_path in enumerate(image_paths):
        img = Image.open(img_path)
        W, H = img.size
        scale = W / mast3r_size

        if focals.shape[1] == 1:
            focal_mast3r = float(focals[i, 0])
            fx = fy = focal_mast3r * scale
        else:
            fx = float(focals[i, 0]) * scale
            fy = float(focals[i, 1]) * scale

        cx = float(pp[i, 0]) * scale
        cy = float(pp[i, 1]) * scale

        camera = {
            'camera_id': i + 1,
            'model': 'PINHOLE',
            'width': W,
            'height': H,
            'params': [fx, fy, cx, cy]
        }
        cameras.append(camera)

        if i == 0:
            print(f"\nExample camera 0:")
            print(f"  Image size: {W}x{H}")
            print(f"  MASt3R focal: {focal_mast3r:.2f}, pp: ({pp[i,0]:.2f}, {pp[i,1]:.2f})")
            print(f"  Scaled fx={fx:.2f}, fy={fy:.2f}, cx={cx:.2f}, cy={cy:.2f}")
            print(f"  Pose (first row): {poses[i][0]}")

    print(f"\n✓ Extracted {len(cameras)} cameras and {len(poses)} poses")

    pts3d = pts3d.reshape(-1, 3)
    colors = np.ones((len(pts3d), 3)) * 0.5
    
    return pts3d, colors, cameras, poses


# ===== Traditional Method: rotmat2qvec =====
def rotmat2qvec_traditional(R):
    """Traditional Method: Convert rotation matrix to quaternion."""
    R = np.asarray(R, dtype=np.float64)
    trace = np.trace(R)

    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
        w = (R[2, 1] - R[1, 2]) / s
        x = 0.25 * s
        y = (R[0, 1] + R[1, 0]) / s
        z = (R[0, 2] + R[2, 0]) / s
    elif R[1, 1] > R[2, 2]:
        s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
        w = (R[0, 2] - R[2, 0]) / s
        x = (R[0, 1] + R[1, 0]) / s
        y = 0.25 * s
        z = (R[1, 2] + R[2, 1]) / s
    else:
        s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
        w = (R[1, 0] - R[0, 1]) / s
        x = (R[0, 2] + R[2, 0]) / s
        y = (R[1, 2] + R[2, 1]) / s
        z = 0.25 * s

    qvec = np.array([w, x, y, z], dtype=np.float64)
    qvec = qvec / np.linalg.norm(qvec)

    return qvec


# ===== Traditional Method: Save Functions =====
def write_cameras_binary_traditional(cameras, output_file):
    """Traditional Method: Write cameras.bin."""
    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', len(cameras)))

        for i, cam in enumerate(cameras):
            camera_id = cam.get('camera_id', i + 1)
            model_id = 1  # PINHOLE
            width = cam['width']
            height = cam['height']
            params = cam['params']

            f.write(struct.pack('i', camera_id))
            f.write(struct.pack('i', model_id))
            f.write(struct.pack('Q', width))
            f.write(struct.pack('Q', height))

            for param in params[:4]:
                f.write(struct.pack('d', param))


def write_images_binary_traditional(image_paths, cameras, poses, output_file):
    """Traditional Method: Write images.bin."""
    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', len(image_paths)))

        for i, (img_path, pose) in enumerate(zip(image_paths, poses)):
            image_id = i + 1
            camera_id = cameras[i].get('camera_id', i + 1)
            image_name = os.path.basename(img_path)

            R = pose[:3, :3]
            t = pose[:3, 3]
            qvec = rotmat2qvec_traditional(R)
            tvec = t

            f.write(struct.pack('i', image_id))
            for q in qvec:
                f.write(struct.pack('d', float(q)))
            for tv in tvec:
                f.write(struct.pack('d', float(tv)))
            f.write(struct.pack('i', camera_id))
            f.write(image_name.encode('utf-8') + b'\x00')
            f.write(struct.pack('Q', 0))


def write_points3d_binary_traditional(pts3d, colors, output_file):
    """Traditional Method: Write points3D.bin."""
    valid_indices = []
    invalid_count = 0
    
    for i, pt in enumerate(pts3d):
        if not (np.isnan(pt).any() or np.isinf(pt).any()):
            valid_indices.append(i)
        else:
            invalid_count += 1

    with open(output_file, 'wb') as f:
        f.write(struct.pack('Q', len(valid_indices)))

        for idx, point_id in enumerate(valid_indices):
            pt = pts3d[point_id]
            color = colors[point_id]

            f.write(struct.pack('Q', point_id))
            for coord in np.asarray(pt).ravel(): 
                    f.write(struct.pack('d', float(coord)))

            col_int = (color * 255).astype(np.uint8)
            for c in col_int:
                f.write(struct.pack('B', int(c)))

            f.write(struct.pack('d', 0.0))
            f.write(struct.pack('Q', 0))

    if invalid_count > 0:
        print(f"  ⚠ Excluded {invalid_count} invalid points (NaN/Inf)")

    return len(valid_indices)


def save_colmap_reconstruction_traditional(pts3d, colors, cameras, poses, image_paths, output_dir):
    """Traditional Method: Save COLMAP reconstruction."""
    print("\n=== [TRADITIONAL] Saving COLMAP reconstruction ===")

    sparse_dir = Path(output_dir) / 'sparse_traditional' / '0'
    sparse_dir.mkdir(parents=True, exist_ok=True)

    write_cameras_binary_traditional(cameras, sparse_dir / 'cameras.bin')
    print(f"  ✓ Wrote {len(cameras)} cameras")

    write_images_binary_traditional(image_paths, cameras, poses, sparse_dir / 'images.bin')
    print(f"  ✓ Wrote {len(image_paths)} images")

    num_points = write_points3d_binary_traditional(pts3d, colors, sparse_dir / 'points3D.bin')
    print(f"  ✓ Wrote {num_points} 3D points")

    print(f"\n✓ Traditional COLMAP reconstruction saved to {sparse_dir}")

    return sparse_dir

In [3]:
# ============================================================================
# Part 3: テキスト形式での保存 (検証用)
# ============================================================================

def save_colmap_text_for_verification(pts3d, colors, cameras, poses, image_paths, output_dir):
    """検証用: COLMAP形式をテキストで保存"""
    print("\n=== Saving COLMAP in text format for verification ===")
    
    text_dir = Path(output_dir) / 'text_format'
    text_dir.mkdir(parents=True, exist_ok=True)
    
    # cameras.txt
    with open(text_dir / 'cameras.txt', 'w') as f:
        f.write("# Camera list with one line of data per camera:\n")
        f.write("#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
        for cam in cameras:
            f.write(f"{cam['camera_id']} PINHOLE {cam['width']} {cam['height']} ")
            f.write(" ".join([f"{p:.6f}" for p in cam['params']]))
            f.write("\n")
    
    # images.txt
    with open(text_dir / 'images.txt', 'w') as f:
        f.write("# Image list with two lines of data per image:\n")
        f.write("#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
        f.write("#   POINTS2D[] as (X, Y, POINT3D_ID)\n")
        for i, (img_path, pose) in enumerate(zip(image_paths, poses)):
            R = pose[:3, :3]
            t = pose[:3, 3]
            qvec = rotmat2qvec_traditional(R)
            
            f.write(f"{i+1} ")
            f.write(" ".join([f"{q:.6f}" for q in qvec]))
            f.write(" ")
            f.write(" ".join([f"{tv:.6f}" for tv in t]))
            f.write(f" {cameras[i]['camera_id']} {os.path.basename(img_path)}\n")
            f.write("\n")
    
    # points3D.txt
    with open(text_dir / 'points3D.txt', 'w') as f:
        f.write("# 3D point list with one line of data per point:\n")
        f.write("#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)\n")
        
        valid_count = 0
        for i in range(len(pts3d)):
            pt = pts3d[i]
            color = colors[i]
            
            if not (np.isnan(pt).any() or np.isinf(pt).any()):
                col_int = (color * 255).astype(np.uint8)
                f.write(f"{i} ")
                f.write(f"{pt[0]:.6f} {pt[1]:.6f} {pt[2]:.6f} ")
                f.write(f"{col_int[0]} {col_int[1]} {col_int[2]} 0.0\n")
                valid_count += 1
    
    print(f"✓ Saved text format to {text_dir}")
    print(f"  - {len(cameras)} cameras")
    print(f"  - {len(image_paths)} images")
    print(f"  - {valid_count} 3D points")
    
    return text_dir


# ============================================================================
# Part 4: 検証関数
# ============================================================================

def verify_transformation(ground_truth_path, colmap_text_path):
    """COLMAP出力とGround truthを比較"""
    ground_truth_path = Path(ground_truth_path)
    colmap_text_path = Path(colmap_text_path)
    
    # Ground truth読み込み
    gt_points = np.loadtxt(ground_truth_path / 'ground_truth_points3d.txt')
    
    # COLMAP points3D.txt読み込み
    colmap_points_file = colmap_text_path / 'points3D.txt'
    if not colmap_points_file.exists():
        print(f"❌ COLMAP output not found: {colmap_points_file}")
        return False
    
    colmap_points = []
    with open(colmap_points_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) >= 4:
                x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                colmap_points.append([x, y, z])
    
    colmap_points = np.array(colmap_points)
    
    print("\n" + "="*70)
    print("座標変換検証結果")
    print("="*70)
    print(f"\nGround truth点数: {len(gt_points)}")
    print(f"COLMAP点数: {len(colmap_points)}")
    
    if len(colmap_points) == 0:
        print("❌ COLMAP出力に点が見つかりません")
        return False
    
    print("\n【Ground Truth統計】")
    print(f"  平均: {gt_points.mean(axis=0)}")
    print(f"  標準偏差: {gt_points.std(axis=0)}")
    print(f"  最小値: {gt_points.min(axis=0)}")
    print(f"  最大値: {gt_points.max(axis=0)}")
    
    print("\n【COLMAP出力統計】")
    print(f"  平均: {colmap_points.mean(axis=0)}")
    print(f"  標準偏差: {colmap_points.std(axis=0)}")
    print(f"  最小値: {colmap_points.min(axis=0)}")
    print(f"  最大値: {colmap_points.max(axis=0)}")
    
    # 座標の視覚的比較
    print("\n【座標サンプル比較 (最初の5点)】")
    print("Ground Truth:")
    for i in range(min(5, len(gt_points))):
        print(f"  {i}: [{gt_points[i][0]:8.4f}, {gt_points[i][1]:8.4f}, {gt_points[i][2]:8.4f}]")
    print("\nCOLMAP Output:")
    for i in range(min(5, len(colmap_points))):
        print(f"  {i}: [{colmap_points[i][0]:8.4f}, {colmap_points[i][1]:8.4f}, {colmap_points[i][2]:8.4f}]")
    
    # Procrustes解析
    if len(colmap_points) >= len(gt_points):
        try:
            from scipy.spatial import procrustes
            
            n_compare = min(len(gt_points), len(colmap_points))
            gt_compare = gt_points[:n_compare]
            colmap_compare = colmap_points[:n_compare]
            
            gt_centered = gt_compare - gt_compare.mean(axis=0)
            colmap_centered = colmap_compare - colmap_compare.mean(axis=0)
            
            mtx1, mtx2, disparity = procrustes(gt_centered, colmap_centered)
            
            print(f"\n【Procrustes解析】")
            print(f"  比較点数: {n_compare}")
            print(f"  差異度: {disparity:.6f}")
            
            if disparity < 0.01:
                print("  判定: ✓ 座標一致(剛体変換のみ)")
            elif disparity < 0.1:
                print("  判定: ⚠ 概ね一致(小さな差異あり)")
            else:
                print("  判定: ❌ 大きな座標変化を検出")
        except ImportError:
            print("\n⚠ scipy未インストールのため詳細解析をスキップ")
        except Exception as e:
            print(f"\n⚠ Procrustes解析エラー: {e}")
    
    print("="*70 + "\n")
    return True


# ============================================================================
# Part 5: 統合テスト関数
# ============================================================================

def test_traditional_method_with_calibration(num_views=4, num_points=100, 
                                            output_dir='/kaggle/working/calibration_test'):
    """Traditional MethodとキャリブレーションデータでEnd-to-Endテスト"""
    
    print("="*70)
    print("Traditional COLMAP Method + Calibration Test")
    print("="*70)
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # [1] キャリブレーションデータ生成
    print("\n[1/6] キャリブレーションデータ生成中...")
    generator = CalibrationDataGenerator(num_views=num_views, num_points=num_points)
    scene = generator.create_mock_scene(device='cpu')
    
    ground_truth_dir = output_dir / 'ground_truth'
    generator.save_ground_truth(scene, ground_truth_dir)
    
    print(f"  ビュー数: {scene.num_views}")
    print(f"  点数: {scene.num_points}")
    
    # [2] モック画像作成
    print("\n[2/6] モック画像作成中...")
    image_dir = output_dir / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)
    
    image_paths = []
    for i in range(num_views):
        img = Image.new('RGB', (512, 512), color=(128, 128, 128))
        img_path = image_dir / f'view_{i:02d}.jpg'
        img.save(img_path)
        image_paths.append(str(img_path))
    
    print(f"  ✓ {len(image_paths)}枚の画像作成完了")
    
    # [3] Traditional Methodでデータ抽出
    print("\n[3/6] Traditional Methodでデータ抽出中...")
    pts3d, colors, cameras, poses = extract_colmap_data_traditional(
        scene, image_paths, max_points=num_points * num_views
    )
    
    print(f"\n抽出結果:")
    print(f"  pts3d shape: {pts3d.shape}")
    print(f"  colors shape: {colors.shape}")
    print(f"  cameras: {len(cameras)}")
    print(f"  poses: {len(poses)}")
    
    # [4] COLMAP形式で保存 (バイナリ)
    print("\n[4/6] COLMAP形式(バイナリ)で保存中...")
    sparse_dir = save_colmap_reconstruction_traditional(
        pts3d, colors, cameras, poses, image_paths, output_dir
    )
    
    # [5] テキスト形式でも保存 (検証用)
    print("\n[5/6] テキスト形式で保存中...")
    text_dir = save_colmap_text_for_verification(
        pts3d, colors, cameras, poses, image_paths, output_dir
    )
    
    # [6] 検証
    print("\n[6/6] 座標変換検証中...")
    success = verify_transformation(ground_truth_dir, text_dir)
    
    print("\n" + "="*70)
    if success:
        print("✓ テスト完了!")
    else:
        print("⚠ テスト完了 (警告あり)")
    print(f"結果: {output_dir}")
    print(f"  Ground Truth: {ground_truth_dir}")
    print(f"  COLMAP Binary: {sparse_dir}")
    print(f"  COLMAP Text: {text_dir}")
    print("="*70)
    
    return scene, pts3d, colors, cameras, poses


# ============================================================================
# 使用例
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("Traditional COLMAP Method + Calibration Test (修正版)")
    print("="*70)
    print("\n使用方法:")
    print("  scene, pts3d, colors, cameras, poses = test_traditional_method_with_calibration()")
    print("\nこれにより座標変換が正しく行われているか検証できます。")

Traditional COLMAP Method + Calibration Test (修正版)

使用方法:
  scene, pts3d, colors, cameras, poses = test_traditional_method_with_calibration()

これにより座標変換が正しく行われているか検証できます。


In [4]:
scene, pts3d, colors, cameras, poses = test_traditional_method_with_calibration(
    num_views=4,
    num_points=100,
    output_dir='/kaggle/working/calibration_test'
)

Traditional COLMAP Method + Calibration Test

[1/6] キャリブレーションデータ生成中...
✓ Ground truth saved to /kaggle/working/calibration_test/ground_truth
  ビュー数: 4
  点数: 92

[2/6] モック画像作成中...
  ✓ 4枚の画像作成完了

[3/6] Traditional Methodでデータ抽出中...

=== [TRADITIONAL] Extracting COLMAP-compatible data ===
pts_all type: <class 'list'>
pts_all is a list with 4 elements
First element type: <class 'torch.Tensor'>
First element shape: torch.Size([92, 3])
pts_all shape after conversion: torch.Size([4, 92, 3])
✓ Extracted 4 3D points from 4 images
Extracting camera parameters...
Retrieved camera-to-world poses: shape (4, 4, 4)
Converted to world-to-camera poses for COLMAP
Focals shape: (4, 1)
Principal points shape: (4, 2)

Example camera 0:
  Image size: 512x512
  MASt3R focal: 500.00, pp: (256.00, 256.00)
  Scaled fx=1142.86, fy=1142.86, cx=585.14, cy=585.14
  Pose (first row): [-0. -1. -0. -0.]

✓ Extracted 4 cameras and 4 poses

抽出結果:
  pts3d shape: (368, 3)
  colors shape: (368, 3)
  cameras: 4
  poses: 4

[